In [1]:
"""
Build and audit public transport indexes for use in a SUMO/TraCI simulation.

The script links SUMO public transport stops, routes, and vehicles with the
<ride> elements of a SUMO population file. It then exports searchable indexes
and diagnostic files that can be reused by a TraCI controller.

Designed to be executed directly in a Jupyter notebook cell.
"""

from pathlib import Path
import xml.etree.ElementTree as ET
from collections import defaultdict, Counter
import json
import pickle
import csv
from bisect import bisect_left


# ============================================================
# INPUT AND OUTPUT PATHS
# ============================================================
BASE_DIR = Path(".")

POP_ROU = BASE_DIR / "../5-vehicles/population_all_with_vtypes.rou.xml"
PT_STOPS = BASE_DIR / "../3-public_transport/gtfs_pt_stops.add.xml"
PT_VEHICLES = BASE_DIR / "../3-public_transport/gtfs_pt_vehicles_colored.add.xml"

OUT_DIR = BASE_DIR / "pt_index_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Population file :", POP_ROU.resolve())
print("PT stops file   :", PT_STOPS.resolve())
print("PT vehicles file:", PT_VEHICLES.resolve())
print("Output directory:", OUT_DIR.resolve())

# Fail early when an input file is missing.
for path in [POP_ROU, PT_STOPS, PT_VEHICLES]:
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path.resolve()}")


# ============================================================
# HELPERS
# ============================================================
def local_name(tag: str) -> str:
    """Return an XML tag without its optional namespace."""
    return tag.split("}", 1)[-1] if "}" in tag else tag


def to_float(value, default=None):
    """Convert a value to float and return a default when conversion fails."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def first_line(lines_attribute: str) -> str:
    """Return the first line identifier from a SUMO 'lines' attribute."""
    if not lines_attribute:
        return ""
    return str(lines_attribute).split()[0]


# ============================================================
# LOAD PUBLIC TRANSPORT STOPS
# ============================================================
def load_bus_stops(stops_path: Path):
    """Load SUMO bus stops and report duplicate stop identifiers."""
    root = ET.parse(stops_path).getroot()

    stop_info = {}
    duplicates = []

    for element in root.iter():
        if local_name(element.tag) != "busStop":
            continue

        stop_id = element.get("id")
        if stop_id in stop_info:
            duplicates.append(stop_id)

        stop_info[stop_id] = {
            "id": stop_id,
            "name": element.get("name", ""),
            "lane": element.get("lane", ""),
            "startPos": element.get("startPos", ""),
            "endPos": element.get("endPos", ""),
        }

    return stop_info, duplicates


stop_info, duplicated_stop_ids = load_bus_stops(PT_STOPS)
name_counter = Counter(stop["name"] for stop in stop_info.values())

print("\n=== PUBLIC TRANSPORT STOPS ===")
print("Bus stops                 :", len(stop_info))
print("Duplicate stop IDs        :", len(duplicated_stop_ids))
print("Repeated stop names       :", sum(count > 1 for count in name_counter.values()))
print("Most repeated stop names  :", name_counter.most_common(10))


# ============================================================
# LOAD PUBLIC TRANSPORT ROUTES AND VEHICLES
# ============================================================
def load_pt_routes_and_vehicles(vehicles_path: Path):
    """Load SUMO public transport routes and scheduled vehicles."""
    root = ET.parse(vehicles_path).getroot()

    routes = {}
    vehicles = []

    for child in root:
        tag = local_name(child.tag)

        if tag == "route":
            route_id = child.get("id")
            stops = []

            for sub_element in child:
                if local_name(sub_element.tag) != "stop":
                    continue

                stops.append(
                    {
                        "busStop": sub_element.get("busStop"),
                        # gtfs2pt.py stores 'until' relative to vehicle departure.
                        "until": to_float(sub_element.get("until"), default=0.0),
                        "duration": to_float(sub_element.get("duration"), default=0.0),
                    }
                )

            routes[route_id] = stops

        elif tag == "vehicle":
            vehicles.append(
                {
                    "vehicle_id": child.get("id"),
                    "route_id": child.get("route"),
                    "depart": to_float(child.get("depart"), default=0.0),
                    "line": child.get("line", ""),
                    "type": child.get("type", ""),
                }
            )

    return routes, vehicles


routes, vehicles = load_pt_routes_and_vehicles(PT_VEHICLES)
missing_vehicle_routes = [vehicle for vehicle in vehicles if vehicle["route_id"] not in routes]

print("\n=== PUBLIC TRANSPORT ROUTES AND VEHICLES ===")
print("Routes                     :", len(routes))
print("Vehicles                   :", len(vehicles))
print("Vehicles with missing route:", len(missing_vehicle_routes))


# ============================================================
# BUILD PASSAGE AND BOARDING INDEXES
# ============================================================
def build_pt_indexes(routes, vehicles):
    """
    Build two chronological indexes.

    stop_line_passages[(stop_id, line)] lists every vehicle passage at a stop.
    future_boardings[(from_stop, to_stop, line)] lists valid downstream trips.
    """
    stop_line_passages = defaultdict(list)
    future_boardings = defaultdict(list)

    for vehicle in vehicles:
        route_id = vehicle["route_id"]
        vehicle_id = vehicle["vehicle_id"]
        line = vehicle["line"]
        departure_time = vehicle["depart"]

        stops = routes.get(route_id)
        if not stops:
            continue

        # Index each passage of the vehicle at a stop.
        for stop_index, stop in enumerate(stops):
            stop_id = stop["busStop"]
            passage_time = departure_time + stop["until"]

            stop_line_passages[(stop_id, line)].append(
                {
                    "time": passage_time,
                    "vehicle_id": vehicle_id,
                    "route_id": route_id,
                    "line": line,
                    "stop_id": stop_id,
                    "stop_index": stop_index,
                }
            )

        # Index every valid boarding-to-alighting pair in route order.
        for from_index, from_stop_data in enumerate(stops):
            from_stop = from_stop_data["busStop"]
            boarding_time = departure_time + from_stop_data["until"]

            for to_index in range(from_index + 1, len(stops)):
                to_stop_data = stops[to_index]
                to_stop = to_stop_data["busStop"]
                alighting_time = departure_time + to_stop_data["until"]

                future_boardings[(from_stop, to_stop, line)].append(
                    {
                        "boarding_time": boarding_time,
                        "alighting_time": alighting_time,
                        "vehicle_id": vehicle_id,
                        "route_id": route_id,
                        "line": line,
                        "from_stop": from_stop,
                        "to_stop": to_stop,
                        "from_index": from_index,
                        "to_index": to_index,
                    }
                )

    # Sorted lists allow fast binary searches during the simulation.
    for passages in stop_line_passages.values():
        passages.sort(key=lambda item: item["time"])

    for boardings in future_boardings.values():
        boardings.sort(key=lambda item: item["boarding_time"])

    return dict(stop_line_passages), dict(future_boardings)


stop_line_passages, future_boardings = build_pt_indexes(routes, vehicles)

n_stop_line_rows = sum(len(passages) for passages in stop_line_passages.values())
n_boarding_rows = sum(len(boardings) for boardings in future_boardings.values())

print("\n=== INDEXES BUILT ===")
print("Unique (stop, line) keys          :", len(stop_line_passages))
print("Unique (from, to, line) keys      :", len(future_boardings))
print("Indexed stop-line passages        :", n_stop_line_rows)
print("Indexed boarding alternatives     :", n_boarding_rows)


# ============================================================
# QUERY FUNCTIONS FOR TRACI
# ============================================================
def next_passage_at_stop(stop_id: str, line: str, current_time: float):
    """Return the next passage of a line at a specific stop."""
    passages = stop_line_passages.get((stop_id, line), [])
    times = [passage["time"] for passage in passages]
    index = bisect_left(times, current_time)

    if index >= len(passages):
        return None

    return passages[index]


def next_boarding(from_stop: str, to_stop: str, line: str, current_time: float):
    """Return the next vehicle serving the requested stop pair and line."""
    candidates = future_boardings.get((from_stop, to_stop, line), [])
    times = [candidate["boarding_time"] for candidate in candidates]
    index = bisect_left(times, current_time)

    if index >= len(candidates):
        return None

    return candidates[index]


def has_future_boarding(from_stop: str, to_stop: str, line: str, current_time: float) -> bool:
    """Check whether a valid future boarding option exists."""
    return next_boarding(from_stop, to_stop, line, current_time) is not None


# ============================================================
# LOAD RIDES FROM THE SUMO POPULATION
# ============================================================
def iter_population_rides(population_path: Path):
    """Yield every <ride> element together with its parent person ID."""
    current_person = None

    # Streaming parsing avoids loading the complete population XML into memory.
    for event, element in ET.iterparse(population_path, events=("start", "end")):
        tag = local_name(element.tag)

        if event == "start" and tag == "person":
            current_person = element.get("id")

        elif event == "end" and tag == "ride":
            yield {
                "person_id": current_person,
                "from_stop": element.get("fromBusStop"),
                "to_stop": element.get("busStop"),
                "line": first_line(element.get("lines", "")),
            }
            element.clear()

        elif event == "end" and tag == "person":
            current_person = None
            element.clear()


rides = list(iter_population_rides(POP_ROU))

print("\n=== POPULATION RIDES ===")
print("Ride elements              :", len(rides))
print("Distinct lines used by rides:", len(Counter(ride["line"] for ride in rides)))


# ============================================================
# AUDIT POPULATION RIDES
# ============================================================
audit_rows = []
unmatched_rows = []

for ride in rides:
    person_id = ride["person_id"]
    from_stop = ride["from_stop"]
    to_stop = ride["to_stop"]
    line = ride["line"]

    from_stop_exists = from_stop in stop_info
    to_stop_exists = to_stop in stop_info
    stop_line_exists = (from_stop, line) in stop_line_passages
    boarding_exists = (from_stop, to_stop, line) in future_boardings

    first_candidate = None
    if boarding_exists:
        first_candidate = future_boardings[(from_stop, to_stop, line)][0]

    row = {
        "person_id": person_id,
        "from_stop": from_stop,
        "from_name": stop_info.get(from_stop, {}).get("name", ""),
        "to_stop": to_stop,
        "to_name": stop_info.get(to_stop, {}).get("name", ""),
        "line": line,
        "from_stop_exists": from_stop_exists,
        "to_stop_exists": to_stop_exists,
        "stop_line_exists": stop_line_exists,
        "boarding_exists": boarding_exists,
        "first_boarding_time": "" if first_candidate is None else first_candidate["boarding_time"],
        "first_alighting_time": "" if first_candidate is None else first_candidate["alighting_time"],
        "first_vehicle_id": "" if first_candidate is None else first_candidate["vehicle_id"],
    }

    audit_rows.append(row)

    if not (from_stop_exists and to_stop_exists and stop_line_exists and boarding_exists):
        unmatched_rows.append(row)


print("\n=== AUDIT RESULTS ===")
print("Rides with missing origin stop       :", sum(not row["from_stop_exists"] for row in audit_rows))
print("Rides with missing destination stop  :", sum(not row["to_stop_exists"] for row in audit_rows))
print("Rides without a stop-line passage    :", sum(not row["stop_line_exists"] for row in audit_rows))
print("Rides without a valid future boarding:", sum(not row["boarding_exists"] for row in audit_rows))


# ============================================================
# EXPORT RESULTS
# ============================================================
summary = {
    "input_files": {
        "population": str(POP_ROU),
        "pt_stops": str(PT_STOPS),
        "pt_vehicles": str(PT_VEHICLES),
    },
    "n_bus_stops": len(stop_info),
    "n_duplicated_stop_ids": len(duplicated_stop_ids),
    "n_repeated_stop_names": sum(count > 1 for count in name_counter.values()),
    "top_repeated_stop_names": name_counter.most_common(20),
    "n_routes": len(routes),
    "n_vehicles": len(vehicles),
    "n_missing_vehicle_routes": len(missing_vehicle_routes),
    "n_stop_line_keys": len(stop_line_passages),
    "n_future_boarding_keys": len(future_boardings),
    "n_stop_line_rows": n_stop_line_rows,
    "n_future_boarding_rows": n_boarding_rows,
    "n_population_rides": len(rides),
    "n_unmatched_rides": len(unmatched_rows),
    "n_from_stop_missing": sum(not row["from_stop_exists"] for row in audit_rows),
    "n_to_stop_missing": sum(not row["to_stop_exists"] for row in audit_rows),
    "n_stop_line_missing": sum(not row["stop_line_exists"] for row in audit_rows),
    "n_boarding_missing": sum(not row["boarding_exists"] for row in audit_rows),
}

with (OUT_DIR / "pt_audit_summary.json").open("w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

# Export the complete ride audit.
with (OUT_DIR / "pt_rides_audit.csv").open("w", encoding="utf-8", newline="") as file:
    fieldnames = list(audit_rows[0].keys()) if audit_rows else []
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    if fieldnames:
        writer.writeheader()
        writer.writerows(audit_rows)

# Export only rides that could not be matched to a valid PT service.
with (OUT_DIR / "pt_unmatched_rides.csv").open("w", encoding="utf-8", newline="") as file:
    fieldnames = list(audit_rows[0].keys()) if audit_rows else []
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    if fieldnames:
        writer.writeheader()
        writer.writerows(unmatched_rows)

# Pickle preserves the nested Python structures used by the TraCI controller.
with (OUT_DIR / "stop_line_passages.pkl").open("wb") as file:
    pickle.dump(stop_line_passages, file)

with (OUT_DIR / "future_boardings.pkl").open("wb") as file:
    pickle.dump(future_boardings, file)

with (OUT_DIR / "stop_info.pkl").open("wb") as file:
    pickle.dump(stop_info, file)

print("\n=== GENERATED FILES ===")
for path in sorted(OUT_DIR.iterdir()):
    print("-", path)


# ============================================================
# EXAMPLE TRACI USAGE
# ============================================================
print("\n=== EXAMPLE TRACI LOGIC ===")
print(
    """
# During the simulation:
current_time = traci.simulation.getTime()

# For a person waiting for public transport:
from_stop = "..."
to_stop = "..."
line = "..."

candidate = next_boarding(from_stop, to_stop, line, current_time)

if candidate is None:
    # No future vehicle serves this stop pair on the requested line.
    decision = "correct_immediately"
else:
    # A future boarding exists and can be compared with a waiting threshold.
    next_time = candidate["boarding_time"]
    wait_time = next_time - current_time
    decision = "keep_waiting"
"""
)

Population file : C:\Users\ngale\Documents\Documents\Simulation\SUMO-demand2traffic\2-SUMO\5-vehicles\population_all_with_vtypes.rou.xml
PT stops file   : C:\Users\ngale\Documents\Documents\Simulation\SUMO-demand2traffic\2-SUMO\3-public_transport\gtfs_pt_stops.add.xml
PT vehicles file: C:\Users\ngale\Documents\Documents\Simulation\SUMO-demand2traffic\2-SUMO\3-public_transport\gtfs_pt_vehicles_colored.add.xml
Output directory: C:\Users\ngale\Documents\Documents\Simulation\SUMO-demand2traffic\2-SUMO\6-simulation\pt_index_out

=== PUBLIC TRANSPORT STOPS ===
Bus stops                 : 1036
Duplicate stop IDs        : 0
Repeated stop names       : 446
Most repeated stop names  : [('Place de Verdun', 11), ("L'Aubreçay", 5), ('Lycée Valin', 5), ('Le Payaud', 5), ('Collège de Beauregard', 5), ('Jules Ferry', 4), ('Normandin', 4), ('Mitterrand', 4), ('La Ribotelière', 4), ('Belle Croix', 4)]

=== PUBLIC TRANSPORT ROUTES AND VEHICLES ===
Routes                     : 240
Vehicles                

In [2]:
from pathlib import Path

print('Index PT exists :', Path('pt_index_out/future_boardings.pkl').exists())
print('Population exists :', Path('../5-vehicles/population_all_with_vtypes.rou.xml').exists())
print('SUMO config exists :', Path('sim.sumocfg').exists())

Index PT exists : True
Population exists : True
SUMO config exists : True


In [4]:
# Run the event-driven TraCI controller for all persons, monitor PT waiting,
# detect persistent stranded passengers, and apply the configured fallback strategy.
!python traci_controller_strategy.py \
    --sumocfg sim.sumocfg \
    --population ../5-vehicles/population_all_with_vtypes.rou.xml \
    --index-dir pt_index_out \
    --output-dir traci_strategy_event \
    --all-persons \
    --check-every 600 \
    --monitor-every 60 \
    --stranded-confirmation-time 600 \
    --facilities-csv "../2-POI's/facilities2sumo_multimode.csv" \
    --sumo-binary sumo-gui \
    --end 108000 \
    --print-every-scan 3 \
    --suppress-sumo-warnings \
    --seed 1234 \
    --close-gui-on-end


=== GENERIC CONTROLLER — FULL POPULATION ===
Indexed PT plans: 2872 persons
SUMO network         : C:\Users\ngale\Documents\Documents\Simulation\SUMO-demand2traffic\2-SUMO\1-network\cda_la_rochelle.net.xml
Facilities CSV       : C:\Users\ngale\Documents\Documents\Simulation\SUMO-demand2traffic\2-SUMO\2-POI's\facilities2sumo_multimode.csv
Mapped POIs          : 15614
Ped-to-pass edge map : 1544 pedestrian edges
SUMO command: C:\Program Files (x86)\Eclipse\Sumo\bin\sumo-gui.exe -c sim.sumocfg --no-step-log true --no-warnings true --log traci_strategy_event\sumo_full.log --log.timestamps true --error-log traci_strategy_event\sumo_errors.log --start --quit-on-end --end 108000.0 --seed 1234
[OUTPUT] C:\Users\ngale\Documents\Documents\Simulation\SUMO-demand2traffic\2-SUMO\6-simulation\traci_strategy_event\focused_original_plans.xml
[OUTPUT] C:\Users\ngale\Documents\Documents\Simulation\SUMO-demand2traffic\2-SUMO\6-simulation\traci_strategy_event\focused_runtime_trace.csv
[OUTPUT] C:\Users\n

c:\Users\ngale\Documents\Documents\Simulation\SUMO-demand2traffic\2-SUMO\6-simulation\traci_controller_strategy.py:491: UserWarning: Call to deprecated function getNextStops.
  next_stops = traci.vehicle.getNextStops(veh_id)
